# 📦 Project Task: SiCepat Ekspres — First-Mile Logistics Analytics
## Data Cleaning, Feature Engineering & Exploratory Data Analysis

**Module 2 — Python for Data Analysis | Purwadhika Digital Technology School**

---

> ⚠️ **Jangan di-run dulu.** Copy notebook ini terlebih dahulu, baru kerjakan di file copy-an kamu.

---

### Konteks Bisnis
SiCepat Ekspres merebut hati jutaan seller UMKM melalui layanan First-Mile Pickup gratis dan paket murah HALU. Namun pertumbuhan masif ini memunculkan dua masalah kritis: **Phantom Pickup** oleh kurir yang manipulasi data untuk hindari denda KPI, dan **revenue leakage** miliaran rupiah akibat seller yang sengaja memperkecil berat paket di aplikasi.

Kamu berperan sebagai Data Analyst di tim **Business Intelligence SiCepat** yang diminta untuk membersihkan data operasional, mengidentifikasi pola Phantom Pickup dan manipulasi berat, serta memberikan rekomendasi berbasis data untuk SLA enforcement dan revenue recovery.

**Dataset (3 tabel):**
- `sicepat_sellers.csv` — 15.000 baris (Dimensi Seller)
- `sicepat_services.csv` — 5 baris (Dimensi Layanan)
- `sicepat_pickups.csv` — 300.000 baris (Fakta Pickup & Berat)

---

### ⚠️ Catatan Penting
- Task ini **open-ended** — tidak ada satu jawaban yang mutlak benar
- Yang dinilai: **ketepatan keputusan**, **kualitas justifikasi**, dan **kedalaman analisis**
- Setiap keputusan di Data Cleaning & Feature Engineering **wajib disertai penjelasan** di markdown cell
- EDA dikerjakan **tanpa visualisasi** — gunakan pandas aggregation, filtering, sorting, dan merge

---

### 🚨 Business Context Error (Wajib Diinvestigasi)
> **~10.199 transaksi** dengan `pickup_status = 'Success'` memiliki `pickup_time` yang terjadi **SEBELUM** `request_time`.  
> Ini adalah **Phantom Pickup**: kurir menekan tombol 'Pickup Selesai' sebelum seller bahkan memanggil.  
> Logika mustahil secara operasional — identifikasi, kuantifikasi, dan rekomendasikan mekanisme deteksi otomatis.

---
## 0. Import & Load Data

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

In [2]:
# Load semua dataset
# Sesuaikan path dengan lokasi file kamu
df_sellers_raw  = pd.read_csv('sicepat_sellers.csv')
df_services_raw = pd.read_csv('sicepat_services.csv')
df_pickups_raw  = pd.read_csv('sicepat_pickups.csv')

# Buat copy untuk dikerjakan
df_sellers  = df_sellers_raw.copy()
df_services = df_services_raw.copy()
df_pickups  = df_pickups_raw.copy()


sellers  : (15000, 4)
services : (5, 2)
pickups  : (300000, 9)


---
## 2. Data Cleaning

### 2.1 Eksplorasi Awal (Wajib)

Lakukan eksplorasi menyeluruh pada **ketiga tabel** sebelum membersihkan data apapun.

In [3]:

print(f'sellers  : {df_sellers.shape}')
print(f'services : {df_services.shape}')
print(f'pickups  : {df_pickups.shape}')

sellers  : (15000, 4)
services : (5, 2)
pickups  : (300000, 9)


<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   service_code  5 non-null      str  
 1   service_name  5 non-null      str  
dtypes: str(2)
memory usage: 271.0 bytes


In [5]:
df_pickups.info()

<class 'pandas.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 9 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   resi_no                  300000 non-null  str    
 1   seller_id                300000 non-null  str    
 2   service_code             300000 non-null  str    
 3   item_category            269991 non-null  str    
 4   request_time             300000 non-null  str    
 5   pickup_time              277552 non-null  str    
 6   stated_weight_kg         300000 non-null  float64
 7   actual_volume_weight_kg  300000 non-null  float64
 8   pickup_status            300000 non-null  str    
dtypes: float64(2), str(7)
memory usage: 44.9 MB


In [6]:
# Tipe data seluruh kolom
print(df_sellers.dtypes,df_services.dtypes,df_pickups.dtypes )

seller_id      str
seller_name    str
city           str
join_date      str
dtype: object service_code    str
service_name    str
dtype: object resi_no                        str
seller_id                      str
service_code                   str
item_category                  str
request_time                   str
pickup_time                    str
stated_weight_kg           float64
actual_volume_weight_kg    float64
pickup_status                  str
dtype: object


In [7]:
df_pickups.head

<bound method NDFrame.head of                resi_no  seller_id service_code item_category  \
0       000SC000000001  SEL-04246       SVC-05         ksmtk   
1       000SC000000002  SEL-04881       SVC-02       Pakaian   
2       000SC000000003  SEL-10176       SVC-02      Skincare   
3       000SC000000004  SEL-05400       SVC-02      Kosmetik   
4       000SC000000005  SEL-08010       SVC-02           NaN   
...                ...        ...          ...           ...   
299995  000SC000299996  SEL-04921       SVC-02      Kosmetik   
299996  000SC000299997  SEL-13537       SVC-03       Pakaian   
299997  000SC000299998  SEL-04746       SVC-03          Baju   
299998  000SC000299999  SEL-01880       SVC-05      Kosmetik   
299999  000SC000300000  SEL-12449       SVC-03       Pakaian   

               request_time                 pickup_time  stated_weight_kg  \
0       2023-08-20 15:00:00  2023-08-20 20:11:58.720564              2.51   
1       2023-08-14 06:00:00  2023-08-15 01:43:1

In [8]:
df_services.head

<bound method NDFrame.head of   service_code service_name
0       SVC-01         HALU
1       SVC-02         BEST
2       SVC-03     SIUNTUNG
3       SVC-04        GOKIL
4       SVC-05     HALU-COD>

In [9]:
df_sellers.head()

,seller_id,seller_name,city,join_date
0,SEL-00001,Toko Seller Sukses 0,Cimahi,2021-11-27
1,SEL-00002,Toko Seller Sukses 1,Makassar,2022-02-19
2,SEL-00003,Toko Seller Sukses 2,Cimahi,2021-06-20
3,SEL-00004,Toko Seller Sukses 3,Bandung,2021-09-18
4,SEL-00005,Toko Seller Sukses 4,Medan,2021-06-09


In [ ]:
# Missing values: jumlah dan persentase per kolom, per tabel
# Tampilkan hanya kolom yang memiliki missing values
# Missing values: jumlah dan persentase per kolom, per tabel
# Tampilkan hanya kolom yang memiliki missing values

print("Total nilai kosong pada tiap kolom :")

print("Missing Value Sellers:")
print(df_sellers.isnull().sum())
na_value1  = df_sellers.isnull().sum()
na_value1.apply(lambda  x:f"{x} - {x/len(df_sellers):.2%}")

print("\nMissing Value Services:")
print(df_services.isnull().sum())

na_value3  = df_pickups.isnull().sum()
print("\nPersentase Missing Value Pickups:")

na_value3.apply(lambda  x:f"{x} - {x/len(df_pickups):.2%}")


Total nilai kosong pada tiap kolom :
Missing Value Sellers:
seller_id      0
seller_name    0
city           0
join_date      0
dtype: int64

Missing Value Services:
service_code    0
service_name    0
dtype: int64

Persentase Missing Value Pickups:


resi_no                         0 - 0.00%
seller_id                       0 - 0.00%
service_code                    0 - 0.00%
item_category              30009 - 10.00%
request_time                    0 - 0.00%
pickup_time                 22448 - 7.48%
stated_weight_kg                0 - 0.00%
actual_volume_weight_kg         0 - 0.00%
pickup_status                   0 - 0.00%
dtype: str

In [ ]:
# Distribusi kolom-kolom kritis
# stated_weight_kg, actual_volume_weight_kg, pickup_status, item_category

# 1. Distribusi untuk kolom numerik (stated_weight_kg & actual_volume_weight_kg)
print("--- Statistik Deskriptif Kolom Numerik ---")
print(df_pickups[['stated_weight_kg', 'actual_volume_weight_kg']].describe())

print("\n--- Distribusi Kolom Kategori: pickup_status ---")
# Menampilkan frekuensi dan persentase pickup_status
status_dist = df_pickups['pickup_status'].value_counts(dropna=False)
status_pct = df_pickups['pickup_status'].value_counts(dropna=False, normalize=True) * 100
print(pd.DataFrame({'Jumlah': status_dist, 'Persentase (%)': status_pct}))

print("\n--- Distribusi Kolom Kategori: item_category ---")
# Menampilkan frekuensi dan persentase item_category (termasuk NaN)
cat_dist = df_pickups['item_category'].value_counts(dropna=False)
cat_pct = df_pickups['item_category'].value_counts(dropna=False, normalize=True) * 100
print(pd.DataFrame({'Jumlah': cat_dist, 'Persentase (%)': cat_pct}))

--- Statistik Deskriptif Kolom Numerik ---
       stated_weight_kg  actual_volume_weight_kg
count         300000.00                300000.00
mean               6.15                     4.15
std               58.34                     2.84
min               -1.50                     0.50
25%                1.61                     2.17
50%                2.75                     3.55
75%                3.88                     4.88
max              999.90                    14.99

--- Distribusi Kolom Kategori: pickup_status ---
               Jumlah  Persentase (%)
pickup_status                        
Success        254992           85.00
Failed          30090           10.03
Rescheduled     14918            4.97

--- Distribusi Kolom Kategori: item_category ---
               Jumlah  Persentase (%)
item_category                        
Baju            45250           15.08
Kosmetik        44985           14.99
baju            30040           10.01
NaN             30009           10.0

In [ ]:
# 1. Investigasi distribusi stated_weight_kg secara menyeluruh
print("=== Ringkasan Statistik Stated Weight (kg) ===")
print(df_pickups["stated_weight_kg"].describe())

# 2. Identifikasi ketiga jenis anomali dan jumlah baris terdampak
negatif = df_pickups[df_pickups["stated_weight_kg"] < 0]
sangat_kecil = df_pickups[
    (df_pickups["stated_weight_kg"] >= 0) & (df_pickups["stated_weight_kg"] < 0.1)
]
sangat_besar = df_pickups[df_pickups["stated_weight_kg"] > 100]

print("\n=== Jumlah Baris per Tipe Anomali ===")
print(f"1. Nilai Negatif (< 0 kg)         : {len(negatif)} baris")
print(f"2. Nilai Sangat Kecil (< 0.1 kg)  : {len(sangat_kecil)} baris")
print(f"3. Nilai Sangat Besar (> 100 kg)  : {len(sangat_besar)} baris")

=== Ringkasan Statistik Stated Weight (kg) ===
count   300000.00
mean         6.15
std         58.34
min         -1.50
25%          1.61
50%          2.75
75%          3.88
max        999.90
Name: stated_weight_kg, dtype: float64

=== Jumlah Baris per Tipe Anomali ===
1. Nilai Negatif (< 0 kg)         : 969 baris
2. Nilai Sangat Kecil (< 0.1 kg)  : 1001 baris
3. Nilai Sangat Besar (> 100 kg)  : 1030 baris


**✍️ Ringkasan Temuan Eksplorasi:**

*(Kolom apa yang bermasalah di setiap tabel, seberapa parah, dan prioritas penanganan kamu)*

> 

---
### 2.2 Kerangka Identifikasi Missing Values

Sebelum menangani missing values pada kolom manapun, identifikasi dulu **jenis missing value-nya**.

| Jenis | Definisi Singkat | Implikasi Penanganan | Contoh di Dataset Ini |
|---|---|---|---|
| **MCAR** *(Missing Completely At Random)* | Nilai kosong tidak berkaitan dengan kolom lain. Pola missing benar-benar acak. | Relatif aman di-impute atau di-drop tanpa bias signifikan. | `item_category` kosong secara acak tanpa pola tertentu. |
| **MAR** *(Missing At Random)* | Nilai kosong berkaitan dengan kolom **lain**, bukan dengan nilai kolom itu sendiri. | Imputation berbasis kolom lain lebih tepat. Drop bisa menyebabkan bias. | `item_category` kosong lebih sering pada seller tertentu yang malas mengisi formulir. |
| **MNAR** *(Missing Not At Random)* | Nilai kosong berkaitan langsung dengan nilai yang seharusnya ada. Ada alasan sistematis. | Imputation apapun berisiko misleading. Perlu keputusan bisnis eksplisit. | `pickup_time` kosong karena transaksi Failed/Rescheduled — nilai kosong itu sendiri adalah informasi operasional. |

> 💡 Justifikasi reasoning kamu lebih penting dari labelnya.

---
### 2.3 Penanganan `item_category`

Kolom ini diisi manual oleh seller di aplikasi e-commerce, menghasilkan **11 varian penulisan** untuk ~6 kategori, ditambah ~10% missing (~30.009 baris).

| Varian Asli | Kategori Standar yang Dimaksud |
|---|---|
| `Baju`, `baju`, `Pakaian`, `Fashion` | Fashion & Pakaian |
| `Kosmetik`, `ksmtk` | Kecantikan |
| `Skincare` | Kecantikan (atau terpisah?) |
| `Elektronik`, `hp` | Elektronik |
| `Sepatu`, `spt` | Alas Kaki |
| `NaN` (~30.009 baris) | Unknown / Perlu keputusan |

> 🧠 **Critical Thinking Prompt:**  
> Apakah 'Kosmetik' dan 'Skincare' benar-benar sama? Keduanya mungkin memiliki profil berat volumetrik berbeda.  
> Pertimbangkan kebutuhan analisis downstream sebelum memutuskan untuk menggabungkan atau memisahkan.

In [5]:
print("Distribusi pickup_status:")
print(df_pickups["pickup_status"].value_counts())
print("\nDistribusi item_category:")
print(df_pickups["item_category"].value_counts(dropna=False))

Distribusi pickup_status:
pickup_status
Success        254992
Failed          30090
Rescheduled     14918
Name: count, dtype: int64

Distribusi item_category:
item_category
Baju          45250
Kosmetik      44985
baju          30040
NaN           30009
Fashion       29942
Pakaian       29874
Skincare      29802
Elektronik    15087
ksmtk         15003
hp            14975
Sepatu        12025
spt            3008
Name: count, dtype: int64


**✍️ Mapping standarisasi yang kamu buat:**
- Varian asli → nilai standar (tuliskan mapping lengkapnya):
- Apakah 'Skincare' digabung dengan 'Kosmetik' atau dipisah? Alasan:
- **Jenis missing value (MCAR / MAR / MNAR):** dan alasan klasifikasi kamu:
- Keputusan penanganan missing values dan alasan:

> 

In [ ]:
mapping_category = {
    "Baju": "Fashion & Pakaian",
    "baju": "Fashion & Pakaian",
    "Pakaian": "Fashion & Pakaian",
    "Fashion": "Fashion & Pakaian",
    "Kosmetik": "Kecantikan & Skincare",
    "ksmtk": "Kecantikan & Skincare",
    "Skincare": "Kecantikan & Skincare",
    "Elektronik": "Elektronik",
    "hp": "Elektronik",
    "Sepatu": "Alas Kaki",
    "spt": "Alas Kaki"
}
df_pickups["item_category_clean"] = df_pickups["item_category"].map(mapping_category)
df_pickups["item_category_clean"] = df_pickups["item_category_clean"].fillna("Unknown")
df_pickups["item_category_clean"].value_counts()

item_category_clean
Fashion & Pakaian        135106
Kecantikan & Skincare     89790
Elektronik                30062
Unknown                   30009
Alas Kaki                 15033
Name: count, dtype: int64

---
### 2.4 Penanganan `stated_weight_kg`

Kolom ini memiliki tiga jenis anomali berbeda yang masing-masing butuh penanganan terpisah:

| Tipe Anomali | Jumlah Baris (approx.) | Kemungkinan Penyebab |
|---|---|---|
| Nilai negatif (< 0) | ~969 baris | Input error seller, bug validasi form |
| Nilai sangat kecil (< 0.1 kg) | ~1.970 baris | Default value yang tidak diubah, atau manipulasi disengaja |
| Nilai sangat besar (> 100 kg) | ~1.030 baris | Salah satuan (gram vs kg), atau paket industri |

> 🧠 **Critical Thinking Prompt:**  
> Nilai stated_weight yang sangat kecil (0.01 kg) mungkin adalah seller yang **sengaja mengecilkan berat** untuk hemat ongkir.  
> Ini adalah potensi revenue leakage yang berbeda dari sekadar input error.  
> Apakah keputusan kamu berbeda jika anomali tersebut berkorelasi dengan `weight_gap` yang besar?

In [11]:
# 1. Investigasi distribusi stated_weight_kg secara menyeluruh
print("=== Ringkasan Statistik Stated Weight (kg) ===")   
print(df_pickups["stated_weight_kg"].describe())

# 2. Identifikasi ketiga jenis anomali dan jumlah baris terdampak
negatif = df_pickups[df_pickups["stated_weight_kg"] < 0]
sangat_kecil = df_pickups[
    (df_pickups["stated_weight_kg"] >= 0) & (df_pickups["stated_weight_kg"] < 0.1)
]
sangat_besar = df_pickups[df_pickups["stated_weight_kg"] > 100]

print("\n=== Jumlah Baris per Tipe Anomali ===")
print(f"1. Nilai Negatif (< 0 kg)         : {len(negatif)} baris")
print(f"2. Nilai Sangat Kecil (< 0.1 kg)  : {len(sangat_kecil)} baris")
print(f"3. Nilai Sangat Besar (> 100 kg)  : {len(sangat_besar)} baris")

=== Ringkasan Statistik Stated Weight (kg) ===
count   300000.00
mean         6.15
std         58.34
min         -1.50
25%          1.61
50%          2.75
75%          3.88
max        999.90
Name: stated_weight_kg, dtype: float64

=== Jumlah Baris per Tipe Anomali ===
1. Nilai Negatif (< 0 kg)         : 969 baris
2. Nilai Sangat Kecil (< 0.1 kg)  : 1001 baris
3. Nilai Sangat Besar (> 100 kg)  : 1030 baris


In [12]:
print(df_pickups.columns.tolist())

['resi_no', 'seller_id', 'service_code', 'item_category', 'request_time', 'pickup_time', 'stated_weight_kg', 'actual_volume_weight_kg', 'pickup_status', 'item_category_clean', 'is_negatif', 'is_sangat_kecil', 'is_sangat_besar', 'is_anomali']


In [15]:
# Investigasi lanjutan: apakah anomali berkorelasi dengan seller, service, atau item_category tertentu?
# 1. Buat penanda (flag) anomali (jika belum ada)
df_pickups['is_negatif'] = df_pickups['stated_weight_kg'] < 0
df_pickups['is_sangat_kecil'] = (df_pickups['stated_weight_kg'] >= 0) & (df_pickups['stated_weight_kg'] < 0.1)
df_pickups['is_sangat_besar'] = df_pickups['stated_weight_kg'] > 100
df_pickups['is_anomali'] = df_pickups['is_negatif'] | df_pickups['is_sangat_kecil'] | df_pickups['is_sangat_besar']

# 2. Cek proporsi anomali berdasarkan Seller (Top 10)
print("=== Proporsi Anomali per Seller (Top 10) ===")
print(df_pickups.groupby('seller_id')['is_anomali'].agg(['count', 'sum', 'mean']).rename(columns={'count': 'total', 'sum': 'anomali', 'mean': 'persentase'}).sort_values(by='anomali', ascending=False).head(10))

# 3. Cek proporsi anomali berdasarkan Service Code
print("\n=== Proporsi Anomali per Layanan (Service Code) ===")
print(df_pickups.groupby('service_code')['is_anomali'].agg(['count', 'sum', 'mean']).rename(columns={'count': 'total', 'sum': 'anomali', 'mean': 'persentase'}).sort_values(by='anomali', ascending=False))

# 4. Cek proporsi anomali berdasarkan Item Category
print("\n=== Proporsi Anomali per Kategori Barang ===")
print(df_pickups.groupby('item_category')['is_anomali'].agg(['count', 'sum', 'mean']).rename(columns={'count': 'total', 'sum': 'anomali', 'mean': 'persentase'}).sort_values(by='anomali', ascending=False))

=== Proporsi Anomali per Seller (Top 10) ===
           total  anomali  persentase
seller_id                            
SEL-00001     14        0        0.00
SEL-00002     24        0        0.00
SEL-00003     25        0        0.00
SEL-00004     18        0        0.00
SEL-00005     20        0        0.00
SEL-00006     16        0        0.00
SEL-00007     17        0        0.00
SEL-00008     25        0        0.00
SEL-00009     21        0        0.00
SEL-00010     16        0        0.00

=== Proporsi Anomali per Layanan (Service Code) ===
               total  anomali  persentase
service_code                             
SVC-01        150107        0        0.00
SVC-02         45079        0        0.00
SVC-03         29874        0        0.00
SVC-04         14937        0        0.00
SVC-05         60003        0        0.00

=== Proporsi Anomali per Kategori Barang ===
               total  anomali  persentase
item_category                            
Baju           45250  

**✍️ Analisis & Justifikasi per tipe anomali:**
- **Nilai negatif** — hipotesis penyebab, keputusan penanganan, alasan:
- **Nilai sangat kecil (< 0.1 kg)** — apakah ini error atau manipulasi disengaja? Keputusan:
- **Nilai sangat besar (> 100 kg)** — threshold 'wajar' yang kamu pilih dan justifikasinya:
- Apakah keputusan kamu berbeda jika nilai anomali berkorelasi dengan weight_gap besar?

> 

In [ ]:
df_pickups["stated_weight_kg"] = df_pickups["stated_weight_kg"].where(
    df_pickups["stated_weight_kg"].between(0.1, 100), np.nan
)
print("Missing setelah cleaning:", df_pickups["stated_weight_kg"].isnull().sum())

---
### 2.5 Penanganan `pickup_time` yang Kosong

Kolom `pickup_time` memiliki **~22.448 nilai kosong** (~7.5% dari total). Sebelum diisi atau di-drop, investigasi dulu pola missing-nya.

> 🧠 **Critical Thinking Prompt:**  
> `pickup_time` yang kosong pada transaksi Failed adalah **informasi bisnis**, bukan data rusak.  
> Mengisi NaN dengan nilai apapun akan merusak integritas analisis SLA dan Phantom Pickup.  
> Pertahankan NaN, dan pastikan analisis SLA hanya dilakukan pada baris dengan `pickup_time` yang valid.

In [14]:
# 1 & 2. Breakdown jumlah missing pickup_time berdasarkan pickup_status
missing_pickup = df_pickups[df_pickups["pickup_time"].isna()]

status_breakdown = (
    missing_pickup["pickup_status"]
    .value_counts(dropna=False)
    .reset_index(name="missing_count")
)

# Hitung persentase terhadap total missing
status_breakdown["percentage"] = (
    status_breakdown["missing_count"] / len(missing_pickup)
) * 100

print("=== Breakdown Missing pickup_time per pickup_status ===")
print(status_breakdown)

# Cek apakah ADA pickup_status 'Completed' / 'Success' yang pickup_time-nya NaN
is_all_failed_or_rescheduled = missing_pickup["pickup_status"].isin(
    ["Failed", "Rescheduled"]
).all()
print(
    f"\nApakah semua missing pickup_time hanya pada status Failed/Rescheduled? {is_all_failed_or_rescheduled}"
)

=== Breakdown Missing pickup_time per pickup_status ===
  pickup_status  missing_count  percentage
0        Failed          14973       66.70
1   Rescheduled           7475       33.30

Apakah semua missing pickup_time hanya pada status Failed/Rescheduled? True


**✍️ Analisis & Justifikasi:**
- **Jenis missing value (MCAR / MAR / MNAR):** dan alasan klasifikasi kamu:
- Temuan investigasi (apakah 100% missing pada Failed/Rescheduled?):
- Keputusan penanganan (pertahankan NaN / isi placeholder) dan alasan:
- Implikasi keputusan ini terhadap analisis SLA di Section 3 dan 4:

> 

In [19]:
# Konversi Datetime
df_pickups["request_time"] = pd.to_datetime(
    df_pickups["request_time"]
)

df_pickups["pickup_time"] = pd.to_datetime(
    df_pickups["pickup_time"]
)

---
### 2.6 Penanganan Business Logic Error: Phantom Pickup

**Ini adalah anomali paling kritis di dataset ini.** Sebanyak ~10.199 transaksi dengan `pickup_status = 'Success'` memiliki `pickup_time` **SEBELUM** `request_time` — secara logika operasional mustahil: kurir tidak mungkin menjemput sebelum seller memanggil.

| Tipe Phantom Pickup | Deskripsi | Indikasi |
|---|---|---|
| **Phantom Ringan** | Selisih < 1 jam | Kemungkinan clock skew antar server, bukan manipulasi |
| **Phantom Berat** | Selisih ≥ 1 jam (rata-rata ~3 jam) | Indikasi kuat manipulasi data oleh kurir |

> 🚨 **Critical Thinking Prompt:**  
> **JANGAN drop baris Phantom Pickup** — ini adalah bukti operasional yang sangat berharga.  
> Data ini adalah dasar untuk membangun sistem deteksi kurir nakal dan program suspend otomatis.  
> **Flag, pertahankan, dan analisis secara terpisah.**

In [23]:
# Identifikasi Phantom Pickup: Success dengan pickup_time < request_time
# Berapa jumlahnya? Berapa persentase dari seluruh Success?
success_data = df_pickups[
    df_pickups["pickup_status"] == "Success"
]

phantom_data = success_data[
    success_data["pickup_time"] < success_data["request_time"]
]

print("Jumlah Phantom Pickup:")
print(len(phantom_data))

print("\nJumlah Success:")
print(len(success_data))

print("\nPersentase Phantom Pickup:")
print(
    len(phantom_data) /
    len(success_data) *
    100
)

Jumlah Phantom Pickup:
10199

Jumlah Success:
254992

Persentase Phantom Pickup:
3.9997333249670577


In [25]:
# Investigasi distribusi selisih waktu (jam) pada Phantom Pickup
# Hitung: (request_time - pickup_time) dalam jam untuk kasus Phantom
phantom_data = phantom_data.copy()

phantom_data["phantom_gap_hours"] = (
    phantom_data["request_time"] -
    phantom_data["pickup_time"]
).dt.total_seconds() / 3600

print(phantom_data["phantom_gap_hours"].describe())

count   10199.00
mean        2.99
std         1.13
min         1.17
25%         2.17
50%         3.17
75%         3.82
max         4.82
Name: phantom_gap_hours, dtype: float64


In [26]:
# Investigasi lanjutan: apakah Phantom Pickup terkonsentrasi pada seller, kota, atau service tertentu?
# Flag Phantom
df_pickups["is_phantom_pickup"] = (
    (df_pickups["pickup_status"] == "Success") &
    (df_pickups["pickup_time"].notna()) &
    (df_pickups["pickup_time"] < df_pickups["request_time"])
)
print(
    "Phantom ringan:",
    (phantom_data["phantom_gap_hours"] < 1).sum()
)

print(
    "Phantom berat:",
    (phantom_data["phantom_gap_hours"] >= 1).sum()
)

Phantom ringan: 0
Phantom berat: 10199


In [32]:
# Investigasi lanjutan: apakah Phantom Pickup terkonsentrasi pada seller, kota, atau service tertentu?
# Investigasi lanjutan: apakah Phantom Pickup terkonsentrasi pada seller, kota, atau service tertentu?
# Flag Phantom
df_pickups["is_phantom_pickup"] = (
    (df_pickups["pickup_status"] == "Success") &
    (df_pickups["pickup_time"].notna()) &
    (df_pickups["pickup_time"] < df_pickups["request_time"])
)
# Investigasi Phantom Berdasarkan Kota
phantom_city = df_pickups.merge(
    df_sellers[["seller_id", "city"]],
    on="seller_id",
    how="left"
)

phantom_city = phantom_city.groupby("city").agg(
    total_pickup=("resi_no", "count"),
    phantom_pickup=("is_phantom_pickup", "sum")
)

phantom_city["phantom_rate_percent"] = (
    phantom_city["phantom_pickup"] /
    phantom_city["total_pickup"] *
    100
)

phantom_city.sort_values(
    "phantom_rate_percent",
    ascending=False
)
# Investigasi Phantom Berdasarkan Seller
phantom_seller = df_pickups.groupby("seller_id").agg(
    total_pickup=("resi_no", "count"),
    phantom_pickup=("is_phantom_pickup", "sum")
)

phantom_seller["phantom_rate_percent"] = (
    phantom_seller["phantom_pickup"] /
    phantom_seller["total_pickup"] *
    100
)

phantom_seller[
    phantom_seller["total_pickup"] >= 20
].sort_values(
    "phantom_rate_percent",
    ascending=False
).head(10)

,total_pickup,phantom_pickup,phantom_rate_percent
seller_id,,,
SEL-00776,23,5,21.74
SEL-02557,24,5,20.83
SEL-08496,20,4,20.00
SEL-05873,20,4,20.00
SEL-09484,21,4,19.05
SEL-09512,21,4,19.05
SEL-07049,21,4,19.05
SEL-00649,21,4,19.05
SEL-10817,21,4,19.05


**✍️ Analisis & Justifikasi:**
- Jumlah dan persentase Phantom Pickup dari seluruh transaksi Success:
- Distribusi selisih waktu — apakah lebih banyak Phantom Ringan atau Phantom Berat?
- Pola konsentrasi yang ditemukan (kota/seller/service tertentu?):
- Keputusan penanganan — kolom flag apa yang kamu buat? Apakah kamu bedakan Phantom Ringan vs Berat?
- Hipotesis mengapa bug ini bisa terjadi di aplikasi kurir SiGESIT:

> 

In [30]:
# TODO: Buat flag is_phantom_pickup dan simpan ke df_pickups
# Pertimbangkan: apakah perlu flag terpisah untuk Phantom Ringan vs Phantom Berat?
# Investigasi lanjutan: apakah Phantom Pickup terkonsentrasi pada seller, kota, atau service tertentu?
# Flag Phantom
df_pickups["is_phantom_pickup"] = (
    (df_pickups["pickup_status"] == "Success") &
    (df_pickups["pickup_time"].notna()) &
    (df_pickups["pickup_time"] < df_pickups["request_time"])
)

---
### 2.7 Penanganan Duplikat & Integritas Data

In [33]:
# 1. Cek exact duplicates di setiap tabel
print("=== 1. Exact Duplicates ===")
print("df_pickups:", df_pickups.duplicated().sum())
print("df_sellers:", df_sellers.duplicated().sum())
print("df_services:", df_services.duplicated().sum())

=== 1. Exact Duplicates ===
df_pickups: 0
df_sellers: 0
df_services: 0


In [34]:
# 2. Cek duplikat resi_no
print("\n=== 2. Duplikat resi_no ===")
resi_dup_count = df_pickups.duplicated(subset=['resi_no'], keep=False).sum()
print(f"Jumlah baris dengan resi_no terduplikasi: {resi_dup_count}")


=== 2. Duplikat resi_no ===
Jumlah baris dengan resi_no terduplikasi: 0


In [38]:
# 3. Cek service_code di pickups yang tidak ada di services
print("\n=== 3. Orphan service_code ===")
missing_services = df_pickups[
    ~df_pickups['service_code'].isin(df_services['service_code'])
]
print(
    f"Jumlah baris pickup dengan service_code tidak terdaftar: {len(missing_services)}"
)


=== 3. Orphan service_code ===
Jumlah baris pickup dengan service_code tidak terdaftar: 0


In [40]:
# 4. Cek seller_id di pickups yang tidak ada di sellers
print("\n=== 4. Orphan seller_id ===")
missing_sellers = df_pickups[
    ~df_pickups['seller_id'].isin(df_sellers['seller_id'])
]
print(
    f"Jumlah baris pickup dengan seller_id tidak terdaftar: {len(missing_sellers)}"
)


=== 4. Orphan seller_id ===
Jumlah baris pickup dengan seller_id tidak terdaftar: 0


**✍️ Analisis & Justifikasi:**
- Masalah yang ditemukan dan jumlah baris terdampak:
- Hipotesis untuk setiap masalah:
- Keputusan penanganan per masalah:

> 

In [ ]:
# TODO: Implementasi keputusan penanganan masalah integritas


---
## 3. Feature Engineering

### 3.1 Fitur Wajib

Buat 8 kolom berikut. Sertakan penjelasan singkat business value-nya di setiap fitur.

#### ⚙️ `item_category_clean`
*(Sudah dibuat di Section 2.3 — pastikan sudah ada di df_pickups)*

#### ⚙️ `weight_gap_kg`

> 💡 `actual_volume_weight_kg − stated_weight_kg`. Nilai positif = seller *underdeclare* berat.  
> Gunakan `stated_weight_kg` yang **sudah di-clean** dari Section 2.4.

In [ ]:
# TODO: Buat weight_gap_kg
# Business value: mengukur selisih berat yang menjadi dasar estimasi revenue leakage
df_pickups["weight_gap_kg"] = (
    df_pickups["actual_volume_weight_kg"] -
    df_pickups["stated_weight_kg"]
)

#### ⚙️ `is_oversize`

> 💡 Tentukan threshold kamu sendiri untuk mendefinisikan 'oversize'. Justifikasikan berdasarkan distribusi `weight_gap_kg`.

**✍️ Threshold yang kamu pilih dan alasannya:**

> 

In [20]:
# TODO: Buat is_oversize (boolean)
# Lihat distribusi weight_gap_kg terlebih dahulu untuk menentukan threshold
df_pickups["is_oversize"] = pd.Series(
    pd.NA,
    index=df_pickups.index,
    dtype="boolean"
)

valid_weight = df_pickups["stated_weight_kg"].notna()

df_pickups.loc[valid_weight, "is_oversize"] = (
    df_pickups.loc[valid_weight, "actual_volume_weight_kg"] >
    df_pickups.loc[valid_weight, "stated_weight_kg"] * 1.5
)


#### ⚙️ `is_phantom_pickup`
*(Sudah dibuat di Section 2.6 — pastikan sudah ada di df_pickups)*

#### ⚙️ `sla_hours`

> 💡 Hanya dihitung untuk transaksi `Success` yang **bukan** Phantom Pickup.  
> Untuk Failed, Rescheduled, dan Phantom Pickup: isi dengan NaN.

In [27]:
# TODO: Buat sla_hours
# Business value: mengukur durasi pickup dalam jam untuk analisis SLA compliance
df_pickups["sla_hours"] = np.nan
valid_sla = (
    (df_pickups["pickup_status"] == "Success") &
    (df_pickups["is_phantom_pickup"] == False) &
    (df_pickups["pickup_time"].notna())
)
df_pickups.loc[valid_sla, "sla_hours"] = (
    df_pickups.loc[valid_sla, "pickup_time"] - df_pickups.loc[valid_sla, "request_time"]
).dt.total_seconds() / 3600

#### ⚙️ `is_sla_met`

> 💡 SLA SiCepat First-Mile: 1x24 jam (≤ 24 jam). Null untuk transaksi non-Success.

In [28]:
# TODO: Buat is_sla_met (boolean: True jika sla_hours <= 24)
# Business value: indikator utama performa kurir SiGESIT
df_pickups["is_sla_met"] = pd.Series(pd.NA, index=df_pickups.index, dtype="boolean")
success = df_pickups["pickup_status"] == "Success"
df_pickups.loc[success, "is_sla_met"] = (df_pickups.loc[success, "sla_hours"] <= 24)
print(df_pickups["is_sla_met"].value_counts(dropna=False))

is_sla_met
True     244793
<NA>      45008
False     10199
Name: count, dtype: Int64


#### ⚙️ `seller_tenure_days`

> 💡 Tentukan sendiri tanggal referensi yang kamu gunakan dan justifikasikan.

**✍️ Tanggal referensi yang kamu gunakan dan alasannya:**

> 

In [29]:
# TODO: Buat seller_tenure_days di df_sellers
# Business value: indikator maturitas dan rekam jejak seller
df_sellers["join_date"] = pd.to_datetime(df_sellers["join_date"])
reference_date = pd.Timestamp("2023-12-31")
df_sellers["seller_tenure_days"] = (reference_date - df_sellers["join_date"]).dt.days

#### ⚙️ `service_name`

> 💡 Join df_pickups dengan df_services untuk mendapatkan nama layanan per transaksi.

In [30]:
# TODO: Tambahkan service_name ke df_pickups via merge dengan df_services
df_pickups = df_pickups.merge(df_services[["service_code", "service_name"]], on="service_code", how="left")

---
### 3.2 Fitur Pilihan (Minimal 2)

Pilih minimal 2 dari: `revenue_loss_per_package`, `request_hour`, `pickup_success_rate_per_seller`, `weight_manipulation_flag`, atau fitur buatan sendiri.

#### ⚙️ Fitur Pilihan 1: [Isi nama fitur]

**✍️ Business value dari fitur ini:**

> 

In [32]:
# TODO: Implementasi Fitur Pilihan 1
df_pickups["revenue_loss_per_package"] = (df_pickups["weight_gap_kg"].clip(lower=0) * 1000)


#### ⚙️ Fitur Pilihan 2: [Isi nama fitur]

**✍️ Business value dari fitur ini:**

> 

In [33]:
# TODO: Implementasi Fitur Pilihan 2
df_pickups["request_hour"] = df_pickups["request_time"].dt.hour

---
## 4. Exploratory Data Analysis

> **Aturan:** Semua analisis menggunakan pandas — tanpa visualisasi.  
> Gunakan `.groupby()`, `.agg()`, `.value_counts()`, filtering, sorting, dan **merge antar tabel** saat dibutuhkan.  
> Setiap jawaban **wajib disertai insight** di markdown cell yang tersedia.

---
### 4.1 Analisis Berat & Revenue Leakage

**Soal 1:** Berapa rata-rata, median, dan standar deviasi `stated_weight_kg` vs `actual_volume_weight_kg`? Apa yang bisa disimpulkan dari perbedaan distribusi keduanya?

In [ ]:
# Soal 1


**✍️ Insight:**

> 

**Soal 2:** Berapa total `weight_gap_kg` keseluruhan (hanya baris dengan gap positif)? Berapa estimasi total kerugian dalam Rupiah? Dokumentasikan asumsi tarif per kg yang kamu gunakan.

In [ ]:
# Soal 2
# Dokumentasikan asumsi tarif per kg yang kamu pilih


**✍️ Insight:**

> 

**Soal 3:** Top 10 seller berdasarkan total `weight_gap_kg` kumulatif. Apakah seller-seller ini terkonsentrasi di kota tertentu atau menggunakan layanan tertentu?

In [ ]:
# Soal 3
# Hint: merge df_pickups dengan df_sellers untuk mendapatkan city per seller


**✍️ Insight:**

> 

**Soal 4:** Bandingkan rata-rata `weight_gap_kg` antara layanan **HALU** (promo murah) vs **BEST** (premium). Apakah layanan murah lebih banyak disalahgunakan untuk underdeclare berat?

In [ ]:
# Soal 4


**✍️ Insight:**

> 

---
### 4.2 Analisis SLA & Phantom Pickup

**Soal 5:** Berapa persentase `is_phantom_pickup = True` dari seluruh transaksi Success? Apakah ini tersebar merata atau terkonsentrasi pada seller atau kota tertentu?

In [ ]:
# Soal 5


**✍️ Insight:**

> 

**Soal 6:** Untuk transaksi Success yang **valid** (bukan Phantom Pickup): berapa distribusi `sla_hours`? Berapa persentase yang memenuhi SLA 1x24 jam (`is_sla_met = True`)?

In [ ]:
# Soal 6


**✍️ Insight:**

> 

**Soal 7:** Berapa distribusi `pickup_status` (Success / Failed / Rescheduled) per `service_name`? Apakah layanan tertentu lebih sering mengalami gagal pickup?

In [ ]:
# Soal 7


**✍️ Insight:**

> 

**Soal 8:** Analisis `request_hour`: pada jam berapa request pickup paling banyak terjadi? Apakah ada pola konsentrasi yang bisa digunakan untuk optimasi jadwal kurir?

In [ ]:
# Soal 8
# Hint: ekstrak jam dari request_time menggunakan .dt.hour


**✍️ Insight:**

> 

---
### 4.3 Analisis Seller & Kategori Barang

**Soal 9:** Berapa distribusi jumlah pickup per seller? Identifikasi seller 'power user' (high volume) vs seller biasa. Apakah ada perbedaan pola `weight_gap_kg` antara dua segmen ini?

In [ ]:
# Soal 9
# Tentukan sendiri threshold 'power user' dan justifikasikan


**✍️ Insight:**

> 

**Soal 10:** Berapa distribusi `is_oversize` per `item_category_clean`? Kategori barang apa yang paling sering memiliki berat aktual jauh lebih besar dari berat yang dinyatakan?

In [ ]:
# Soal 10


**✍️ Insight:**

> 

**Soal 11:** Berapa distribusi pickup per `city` (dari tabel sellers)? Kota mana yang paling banyak menghasilkan pickup request, dan kota mana yang memiliki success rate terendah?

In [ ]:
# Soal 11
# Hint: merge df_pickups dengan df_sellers untuk mendapatkan city


**✍️ Insight:**

> 

---
### 4.4 Investigasi Phantom Pickup & SLA Enforcement *(Implicit — Business Sense Required)*

> Kamu diminta **VP Operations SiCepat** untuk menyusun laporan investigasi Phantom Pickup.  
> Temuan ini akan digunakan untuk: (a) menentukan kurir mana yang perlu di-suspend,  
> (b) merancang sistem deteksi otomatis berbasis aturan (*rule-based*), dan  
> (c) mengestimasi dampak terhadap kepuasan seller.

Pilih minimal **2 angle analisis** yang paling relevan untuk menjawab kebutuhan investigasi tersebut.

#### 🔍 Investigasi — Angle 1: [Isi judul]

**✍️ Mengapa kamu memilih angle ini untuk investigasi Phantom Pickup?**

> 

In [ ]:
# Angle 1


**✍️ Insight & Rekomendasi untuk VP Operations:**

> 

#### 🔍 Investigasi — Angle 2: [Isi judul]

**✍️ Mengapa kamu memilih angle ini?**

> 

In [ ]:
# Angle 2


**✍️ Insight & Rule-Based Detection yang kamu usulkan:**

> 

---
### 4.5 Revenue Recovery & Rekomendasi Operasional *(Implicit — Open Ended)*

> Kamu diminta tim **Finance & Operations SiCepat** untuk menyusun laporan Revenue Recovery.  
> Tujuan: mengidentifikasi total kerugian yang bisa di-recover melalui *chargeback* ke platform e-commerce,  
> dan memberikan rekomendasi kebijakan untuk mencegah manipulasi dimensi di masa depan.

> 🧠 **Critical Thinking Prompt:**  
> Tidak semua `weight_gap` adalah manipulasi — beberapa mungkin error pengukuran seller yang jujur.  
> Bagaimana kamu membedakan seller yang **sengaja manipulasi** vs seller yang **tidak tahu cara mengukur dimensi**?  
> Rekomendasi terbaik mempertimbangkan trade-off: terlalu ketat bisa mengusir seller UMKM yang legitimate.

**Ekspektasi minimal:**
- Minimal 3 segmentasi berbeda untuk menganalisis revenue leakage (per service, per kategori barang, per kota seller, dll.)
- Estimasi total nilai kerugian dalam Rupiah dengan asumsi yang jelas dan terdokumentasi
- Minimal 1 rekomendasi kebijakan konkret berbasis data untuk mencegah manipulasi dimensi

**✍️ Pendekatan analisis revenue recovery yang kamu pilih:**

> 

#### 💰 Segmentasi Revenue Leakage 1: [Nama Segmentasi]

In [ ]:
# Segmentasi 1


#### 💰 Segmentasi Revenue Leakage 2: [Nama Segmentasi]

In [ ]:
# Segmentasi 2


#### 💰 Segmentasi Revenue Leakage 3: [Nama Segmentasi]

In [ ]:
# Segmentasi 3


#### 🎯 Estimasi Total Kerugian & Rekomendasi Kebijakan

In [ ]:
# Agregasi estimasi kerugian dari ketiga segmentasi
# Dokumentasikan asumsi tarif per kg yang kamu gunakan


**✍️ Rekomendasi Kebijakan untuk Mencegah Manipulasi Dimensi:**

> 

---
## 5. Export Clean Dataset

In [ ]:
# Gabungkan ketiga tabel menjadi satu dataframe final
# Gunakan LEFT JOIN dengan df_pickups sebagai tabel utama
# Sertakan semua fitur baru yang telah dibuat

# TODO: Implementasi JOIN
# df_final = df_pickups.merge(df_sellers[...], on='seller_id', how='left')
#                      .merge(df_services[...], on='service_code', how='left')

# Export
# df_final.to_csv('sicepat_clean.csv', index=False)
# print(f'Dataset berhasil disimpan: sicepat_clean.csv')
# print(f'Shape final: {df_final.shape}')
# print(f'Kolom baru yang ditambahkan: {[c for c in df_final.columns if c not in df_pickups_raw.columns]}')


---
## 6. Ringkasan & Refleksi

**Keputusan Data Cleaning yang paling challenging dan mengapa:**

> 

**Temuan paling menarik dari EDA (khususnya terkait Phantom Pickup atau revenue leakage):**

> 

**Rekomendasi bisnis utama yang bisa diberikan kepada tim Operations & Finance SiCepat:**

> 